Los tasadores inmobiliarios en Estados Unidos evalúan propiedades para que bancos, compradores y vendedores tomen decisiones financieras bien fundadas. Una tasación incorrecta puede llevar a que un banco otorgue un préstamo por encima del valor real del inmueble, o que un vendedor pierda dinero por subvaluar su propiedad.

El dataset de Ames, Iowa, contiene información detallada sobre más de 80 características de cada casa vendida entre 2006 y 2010: metros cuadrados, calidad de los materiales de construcción, año de construcción, tipo de barrio, cantidad de baños, si tiene garaje, estado del sótano, y muchas más. El objetivo es predecir el precio de venta final de cada propiedad.

Como debemos estimar un precio, estamos ante un problema de regresión.

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

In [2]:
ds = fetch_openml(name="house_prices", as_frame=True)
ds.keys() # ['data', 'target', 'frame', 'categories', 'feature_names', 'target_names', 'DESCR', 'details', 'url']
ds['categories'] #None
ds['feature_names']
ds['target_names'] #'SalePrice'
ds['details'] #información general sobre la base de datos
ds['url']
ds['target']
ds['data'] #dataframe de 1460 filas y 80 atributos
print(ds['DESCR']) #de esta forma se imprime por renglones

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this playground competition's dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence.

With 79 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this competition challenges you to predict the final price of each home.

MSSubClass: Identifies the type of dwelling involved in the sale.	

        20	1-STORY 1946 & NEWER ALL STYLES
        30	1-STORY 1945 & OLDER
        40	1-STORY W/FINISHED ATTIC ALL AGES
        45	1-1/2 STORY - UNFINISHED ALL AGES
        50	1-1/2 STORY FINISHED ALL AGES
        60	2-STORY 1946 & NEWER
        70	2-STORY 1945 & OLDER
        75	2-1/2 STORY ALL AGES
        80	SPLIT OR MULTI-LEVEL
        85	SPLIT FOYER
        90	DUPLEX - ALL STYLES AND AGES
       120	1-STORY PUD (Planned Unit Development) - 1

In [ ]:
np.min(ds['data']['YearBuilt']) #1872
np.max(ds['data']['YearBuilt']) #2010

2010

Hay un rango muy grande de años de construcción, que probablemente influya en los precios de venta, por lo que debemos tener en cuenta este atributo.

In [3]:
ds['target']

0       208500
1       181500
2       223500
3       140000
4       250000
         ...  
1455    175000
1456    210000
1457    266500
1458    142125
1459    147500
Name: SalePrice, Length: 1460, dtype: int64

In [8]:
ds['data']

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,8,2007,WD,Normal
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal


Como ya hemos observado, tenemos un problema de regresión. Leyendo la descripción de los atributos y sus posibles valores en ds['DESCR'], vemos que la mayoría de ellos son categóricos. Algunos de ellos tienen órdenes subyacentes (generalmente de calidad de cosas, por ejemplo la condición del garage), pero muchos otros no tienen un orden claro. Esto plantea una dificultad para estimar los precios por medio de alguna función matemática de las variables, como puede ser una regresión lineal.

Comenzamos clasificando los atributos en numéricos (valores que realmente son números), categóricos con orden implícito y categóricos sin orden claro detrás. No tenemos en cuenta el Id, la clave que identifica a cada instancia, por no tener un significado real en el cálculo de un precio.

Numéricas: MoSold, YrSold, LotFrontage, LotArea, OverallQual, OveralCond, YearBuilt, YearRemodAdd, MasVnrArea, BsmtFinSF1, BsmtFinSF2, BsmtUnfSF, TotalBsmtUnfSF, 1stFlrSF, 2ndFlrSF, LowQualFinSF, GrLivArea, BsmtFullBath, BsmtHalfBath, FullBath, HalfBath, Bedroom, Kitchen, TotRmsAbvGrd, Fireplaces, GarageYrBlt, GarageCars, GarageArea, WoodDeckSF, OpenPorchSF, EnclosedPorch, 3SsnPorch, ScreenPorch, PoolArea, MiscVal.

Categóricas con orden implícito: Street, Alley, Lotshape, Utilities, LandSlope, ExterQual, ExterCond, BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinType2, HeatingQC, CentralAir, KitchenQual, Functional, FireplaceQu, GarageFinish, GarageQual, GarageCond, PavedDrive, PoolQC, Fence.

Categóricas sin orden implícito: el resto.

In [3]:
def get_data():
    ds = fetch_openml(name="house_prices", as_frame=True)
    X = ds['data']
    y = ds['target']
    return X,y

In [4]:
X,y = get_data()

In [5]:
"""
Preprocesamiento completo del Ames Housing Dataset (pedido a Claude especificando las reglas que se detalla
en esta documentación).
(fetch_openml(name="house_prices")), sin incluir el target (SalePrice) ni el Id.
 
Reglas aplicadas:
- ORDINALES: valor uniforme en [0, 1], peor=0, mejor=1.
  NaN / 'NA' -> 0 (tratado como el peor caso, ya que en Ames significa
  "no tiene esa característica": sin sótano, sin garage, etc.)
- NUMÉRICAS: normalización min-max, mínimo -> 0, máximo -> 1.
  NaN (valores faltantes, ej. LotFrontage, MasVnrArea, GarageYrBlt) -> 0,
  aplicado después de escalar (el min/max se calcula ignorando los NaN).
- NOMINALES: one-hot con pd.get_dummies.
    * Condition1/Condition2, Exterior1st/Exterior2nd y
      BsmtFinType1/BsmtFinType2 comparten el mismo dominio de valores
      posibles, así que se fusionan en un único bloque de dummies:
      no importa si el valor apareció en la primera o la segunda columna.
 
NOTA: BsmtFinType1/BsmtFinType2 se tratan acá como NOMINALES (par fusionado),
a pedido explícito de este mensaje, aunque en la conversación previa se
habían definido como ORDINALES. Si esto no era la intención, hay que
quitarlos de NOMINAL_PAIRED_COLUMNS y devolverlos a ORDINAL_ORDERINGS.
"""
 
import pandas as pd
from sklearn.datasets import fetch_openml
 
 
# ---------------------------------------------------------------------------
# Definición de columnas por tipo
# ---------------------------------------------------------------------------
 
ORDINAL_ORDERINGS = {
    "Street": ["Grvl", "Pave"],
    "Alley": ["NA", "Grvl", "Pave"],
    "LotShape": ["IR3", "IR2", "IR1", "Reg"],
    "Utilities": ["ELO", "NoSeWa", "NoSewr", "AllPub"],
    "LandSlope": ["Sev", "Mod", "Gtl"],
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtQual": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtCond": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtExposure": ["NA", "No", "Mn", "Av", "Gd"],
    "HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],
    "CentralAir": ["N", "Y"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "FireplaceQu": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageFinish": ["NA", "Unf", "RFn", "Fin"],
    "GarageQual": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageCond": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "PavedDrive": ["N", "P", "Y"],
    "PoolQC": ["NA", "Fa", "TA", "Gd", "Ex"],
    "Fence": ["NA", "MnWw", "GdWo", "MnPrv", "GdPrv"],
}
 
NUMERIC_COLUMNS = [
    "MoSold", "YrSold", "LotFrontage", "LotArea", "OverallQual", "OverallCond",
    "YearBuilt", "YearRemodAdd", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2",
    "BsmtUnfSF", "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF",
    "GrLivArea", "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath",
    "BedroomAbvGr", "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt",
    "GarageCars", "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch",
    "3SsnPorch", "ScreenPorch", "PoolArea", "MiscVal",
]
 
# Nominales "simples": una columna -> un bloque de dummies
NOMINAL_SINGLE_COLUMNS = [
    "MSSubClass",  # numérica en apariencia (20, 60, 90...) pero es un código
                   # de categoría de vivienda, no una magnitud ordenable
    "MSZoning", "LandContour", "LotConfig", "Neighborhood",
    "BldgType", "HouseStyle", "RoofStyle", "RoofMatl", "MasVnrType",
    "Foundation", "Heating", "Electrical", "GarageType", "MiscFeature",
    "SaleType", "SaleCondition",
]
 
# Pares de columnas que comparten el mismo dominio de valores posibles y
# deben fusionarse en un único bloque de dummies (col_a, col_b, prefijo)
NOMINAL_PAIRED_COLUMNS = [
    ("Condition1", "Condition2", "Condition"),
    ("Exterior1st", "Exterior2nd", "Exterior"),
    ("BsmtFinType1", "BsmtFinType2", "BsmtFinType"),
]
 
 
# ---------------------------------------------------------------------------
# Funciones auxiliares
# ---------------------------------------------------------------------------
 
def _build_uniform_map(categories):
    n = len(categories)
    if n == 1:
        return {categories[0]: 1.0}
    return {cat: i / (n - 1) for i, cat in enumerate(categories)}
 
 
def _encode_ordinal(df, orderings):
    out = pd.DataFrame(index=df.index)
    for col, order in orderings.items():
        if col not in df.columns:
            continue
        mapping = _build_uniform_map(order)
        series = df[col]
        if "NA" in order:
            series = series.fillna("NA")
        encoded = series.map(mapping)
        # Cualquier NaN remanente (categoría inesperada, o NA sin estar en
        # el orden) se fuerza a 0, tal como se pidió explícitamente
        out[col] = encoded.fillna(0.0)
    return out
 
 
def _normalize_numeric(df, columns):
    out = pd.DataFrame(index=df.index)
    for col in columns:
        if col not in df.columns:
            continue
        serie = df[col].astype(float)
        # min/max se calculan ignorando NaN (comportamiento default de pandas)
        min_v, max_v = serie.min(), serie.max()
        if max_v == min_v:
            out[col] = 0.0  # columna constante: se evita división por cero
        else:
            out[col] = (serie - min_v) / (max_v - min_v)
        # Los NaN (valores faltantes, ej. LotFrontage, MasVnrArea,
        # GarageYrBlt) se completan con 0 luego de escalar
        out[col] = out[col].fillna(0.0)
    return out
 
 
def _one_hot_single(df, columns):
    present = [c for c in columns if c in df.columns]
    sub = df[present].copy()
    if "MSSubClass" in sub.columns:
        sub["MSSubClass"] = sub["MSSubClass"].astype(str)
    return pd.get_dummies(sub, columns=present, prefix=present)
 
 
def _one_hot_paired(df, col_a, col_b, prefix):
    """One-hot fusionado para pares tipo Condition1/Condition2: ambas
    columnas comparten el dominio de valores posibles y no interesa
    distinguir en cuál de las dos apareció cada valor."""
    da = pd.get_dummies(df[col_a]) if col_a in df.columns else pd.DataFrame(index=df.index)
    db = pd.get_dummies(df[col_b]) if col_b in df.columns else pd.DataFrame(index=df.index)
 
    combined = da.add(db, fill_value=0)
    combined = (combined > 0).astype(int)
    combined.columns = [f"{prefix}_{c}" for c in combined.columns]
    return combined
 
 
# ---------------------------------------------------------------------------
# Preprocesamiento completo
# ---------------------------------------------------------------------------
 
def preprocess_ames(df):
    """
    Preprocesa el Ames Housing Dataset (df = ds['data'], sin la columna
    target y sin usar 'Id').
 
    Devuelve un DataFrame con:
        - columnas ordinales -> valor uniforme en [0, 1]
        - columnas numéricas -> normalizadas min-max en [0, 1]
        - columnas nominales -> one-hot (get_dummies), con Condition1/2,
          Exterior1st/2nd y BsmtFinType1/2 fusionadas en un solo bloque
    """
    ordinal_part = _encode_ordinal(df, ORDINAL_ORDERINGS)
    numeric_part = _normalize_numeric(df, NUMERIC_COLUMNS)
    nominal_single_part = _one_hot_single(df, NOMINAL_SINGLE_COLUMNS)
 
    paired_parts = [
        _one_hot_paired(df, col_a, col_b, prefix)
        for col_a, col_b, prefix in NOMINAL_PAIRED_COLUMNS
    ]
 
    df_final = pd.concat(
        [ordinal_part, numeric_part, nominal_single_part, *paired_parts],
        axis=1,
    )
    return df_final
 
 
if __name__ == "__main__":
    ds = fetch_openml(name="house_prices", as_frame=True)
    X_raw = ds["data"]
 
    X_processed = preprocess_ames(X_raw)
 
    print("Shape original:", X_raw.shape)
    print("Shape procesado:", X_processed.shape)
    print(X_processed.head())
 

Shape original: (1460, 80)
Shape procesado: (1460, 217)
   Street  Alley  LotShape  Utilities  LandSlope  ExterQual  ExterCond  \
0     1.0    0.0  1.000000        1.0        1.0       0.75        0.5   
1     1.0    0.0  1.000000        1.0        1.0       0.50        0.5   
2     1.0    0.0  0.666667        1.0        1.0       0.75        0.5   
3     1.0    0.0  0.666667        1.0        1.0       0.50        0.5   
4     1.0    0.0  0.666667        1.0        1.0       0.75        0.5   

   BsmtQual  BsmtCond  BsmtExposure  ...  Exterior_Stone  Exterior_Stucco  \
0       0.8       0.6          0.25  ...               0                0   
1       0.8       0.6          1.00  ...               0                0   
2       0.8       0.6          0.50  ...               0                0   
3       0.6       0.8          0.25  ...               0                0   
4       0.8       0.6          0.75  ...               0                0   

   Exterior_VinylSd  Exterior_WdShin

In [6]:
categoricas = [
    "MSSubClass",
    "MSZoning",
    "LandContour",
    "LotConfig",
    "Neighborhood",
    "Condition1",
    "Condition2",
    "BldgType",
    "HouseStyle",
    "RoofStyle",
    "RoofMatl",
    "Exterior1st",
    "Exterior2nd",
    "MasVnrType",
    "Foundation",
    "Heating",
    "Electrical",
    "GarageType",
    "MiscFeature",
    "SaleType",
    "SaleCondition"
]

Para poder trabajar con el orden de los datos categóricos que tienen orden implícito, le pedimos a Claude armar un código que convierta esos órdenes a valores numéricos, normalizados para que todos estén entre 0 y 1. En los casos donde un valor posible es NaN, este se convierte en 0 ya que suele indicar que el inmueble no tiene una determinada característica.

In [7]:
orderings = {
    "Street":        ["Grvl", "Pave"],
    "Alley":         ["NA", "Grvl", "Pave"],
    "LotShape":      ["IR3", "IR2", "IR1", "Reg"],
    "Utilities":     ["ELO", "NoSeWa", "NoSewr", "AllPub"],
    "LandSlope":     ["Sev", "Mod", "Gtl"],
    "ExterQual":     ["Po", "Fa", "TA", "Gd", "Ex"],
    "ExterCond":     ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtQual":      ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtCond":      ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtExposure":  ["NA", "No", "Mn", "Av", "Gd"],
    "BsmtFinType1":  ["NA", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2":  ["NA", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "HeatingQC":     ["Po", "Fa", "TA", "Gd", "Ex"],
    "CentralAir":    ["N", "Y"],
    "KitchenQual":   ["Po", "Fa", "TA", "Gd", "Ex"],
    "Functional":    ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "FireplaceQu":   ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageFinish":  ["NA", "Unf", "RFn", "Fin"],
    "GarageQual":    ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageCond":    ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "PavedDrive":    ["N", "P", "Y"],
    "PoolQC":        ["NA", "Fa", "TA", "Gd", "Ex"],
    "Fence":         ["NA", "MnWw", "GdWo", "MnPrv", "GdPrv"],
}
 
 
# --- 3. Función auxiliar: lista ordenada -> diccionario de valores 0..1 ---
def build_uniform_map(categories):
    n = len(categories)
    if n == 1:
        return {categories[0]: 1.0}
    return {cat: i / (n - 1) for i, cat in enumerate(categories)}
 
 
# --- 4. Construir la nueva tabla ------------------------------------------
df_ord = X.copy()
 
for col, order in orderings.items():
    if col not in df_ord.columns:
        print(f"Aviso: la columna '{col}' no está en el dataset, se omite.")
        continue
 
    mapping = build_uniform_map(order)
    series = df_ord[col]
 
    # Si "NA" es una categoría válida en el orden, los NaN reales de pandas
    # representan esa categoría (ausencia de la característica).
    if "NA" in order:
        series = series.fillna("NA")
 
    df_ord[col] = series.map(mapping)
 
    # Chequeo de categorías inesperadas (no contempladas en el orden)
    n_missing = df_ord[col].isna().sum()
    if n_missing > 0:
        valores_raros = X.loc[df_ord[col].isna(), col].unique()
        print(
            f"Aviso: {col} tiene {n_missing} valores no mapeados. "
            f"Categorías no contempladas: {valores_raros}"
        )
 
# --- 5. Resultado -----------------------------------------------------------
if __name__ == "__main__":
    cols_transformadas = list(orderings.keys())
    print(df_ord[cols_transformadas].head())

   Street  Alley  LotShape  Utilities  LandSlope  ExterQual  ExterCond  \
0     1.0    0.0  1.000000        1.0        1.0       0.75        0.5   
1     1.0    0.0  1.000000        1.0        1.0       0.50        0.5   
2     1.0    0.0  0.666667        1.0        1.0       0.75        0.5   
3     1.0    0.0  0.666667        1.0        1.0       0.50        0.5   
4     1.0    0.0  0.666667        1.0        1.0       0.75        0.5   

   BsmtQual  BsmtCond  BsmtExposure  ...  CentralAir  KitchenQual  Functional  \
0       0.8       0.6          0.25  ...         1.0         0.75         1.0   
1       0.8       0.6          1.00  ...         1.0         0.50         1.0   
2       0.8       0.6          0.50  ...         1.0         0.75         1.0   
3       0.6       0.8          0.25  ...         1.0         0.75         1.0   
4       0.8       0.6          0.75  ...         1.0         0.75         1.0   

   FireplaceQu  GarageFinish  GarageQual  GarageCond  PavedDrive  Po

In [8]:
np.min(df_ord['YrSold']) #2006
np.max(df_ord['YrSold']) #2010
df_ord_con_target = df_ord.copy()
df_ord_con_target['Target'] = y
df_ord_con_target[['YrSold','Target']].groupby(['YrSold']).mean()

,Target
YrSold,
2006,182549.458599
2007,186063.151976
2008,177360.838816
2009,179432.103550
2010,177393.674286


Como el rango de años de venta considerado es muy bajo, no esperábamos un gran impacto de este atributo en el precio promedio, y esto se comprueba con la última consulta realizada.

Ahora procedemos a dividir los datos en desarrollo y evaluación. No se requiere ninguna forma particular de dividir los folds ya que cada instancia corresponde a una venta diferente. Tomaremos un 10% para test y dividiremos el resto en 5 folds.

In [52]:
X_train, X_test, target_train, target_test = train_test_split(df_ord, y, test_size=0.1, random_state=42,shuffle=True)

In [50]:
def regresion_hist_boosting(lista_max_depth,lista_min_samples_leaf,lista_max_features,train_data,train_target):
    pred_val_fold = []
    target_val_fold = []
    kfold = KFold(n_splits=5)
    for prof_max in lista_max_depth:
        for min_leaf in lista_min_samples_leaf:
            for max_feat in lista_max_features:
                model = HistGradientBoostingRegressor(loss='squared_error',max_depth=prof_max,min_samples_leaf=min_leaf,
                                              max_features=max_feat,random_state=42, categorical_features=categoricas)
                r2_prom = np.zeros(2)
                error_relativo_prom = 0
                for train_idx, val_idx in kfold.split(train_data,train_target):
                    X_train_fold, X_val_fold = train_data.iloc[train_idx], train_data.iloc[val_idx]
                    target_train_fold, target_val_fold = train_target.iloc[train_idx], train_target.iloc[val_idx]
                    model.fit(X_train_fold,target_train_fold)
                    pred_val_fold = model.predict(X_val_fold)
                    error_relativo_fold = np.mean(
                                            np.abs(pred_val_fold - target_val_fold) / target_val_fold
                                    )
                    error_relativo_prom += error_relativo_fold
                    r2_prom += np.array([r2_score(target_val_fold, pred_val_fold)])
                r2_prom = r2_prom/5
                error_relativo_prom /= 5
                print('etapa')
                print(prof_max,min_leaf,max_feat,'r2 promedio = ',r2_prom,error_relativo_prom)
    print(pred_val_fold)
    print(target_val_fold)

In [51]:
lista_max_depth = [3,5,7,None]
lista_min_samples_leaf = [1,5,25,50]
lista_max_features = [0.25,0.5,0.75,1.0]
regresion_hist_boosting(lista_max_depth,lista_min_samples_leaf,lista_max_features,X_train,target_train)

#la última combinación de hiperparámetros utilizada tiene un error absoluto medio aproximado de 27048.
#Es un buen inicio pero hay que mejorarlo.

etapa
3 1 0.25 r2 promedio =  [0.88919211 0.88919211] 0.09273992660247408
etapa
3 1 0.5 r2 promedio =  [0.88606261 0.88606261] 0.09324552846091702
etapa
3 1 0.75 r2 promedio =  [0.8924835 0.8924835] 0.09480341607882294
etapa
3 1 1.0 r2 promedio =  [0.88922175 0.88922175] 0.09454508899444951
etapa
3 5 0.25 r2 promedio =  [0.88923397 0.88923397] 0.093896837793565
etapa
3 5 0.5 r2 promedio =  [0.88792969 0.88792969] 0.0934575847762076
etapa
3 5 0.75 r2 promedio =  [0.89101285 0.89101285] 0.09443797781200787
etapa
3 5 1.0 r2 promedio =  [0.88842348 0.88842348] 0.09535181128961082
etapa
3 25 0.25 r2 promedio =  [0.88673331 0.88673331] 0.09288709443721208
etapa
3 25 0.5 r2 promedio =  [0.88757464 0.88757464] 0.09250217373717604
etapa
3 25 0.75 r2 promedio =  [0.88421601 0.88421601] 0.09386115512283197
etapa
3 25 1.0 r2 promedio =  [0.88410837 0.88410837] 0.09491047911729217
etapa
3 50 0.25 r2 promedio =  [0.87684824 0.87684824] 0.09464907143454984
etapa
3 50 0.5 r2 promedio =  [0.87714683 0.

Dado que el precio de venta de un inmueble depende de sus características registradas, es de esperar que si dos inmuebles tienen valores muy similares en los features también tengan precios de venta parecidos. Por eso probaremos una estrategia de regresión con k-vecinos más cercanos. El desafío es definir la distancia entre dos inmuebles, ya que los distintos atributos no tienen necesariamente la misma importancia ni los mismos valores posibles. Para los atributos numéricos implementaremos una normalización para que tomen valores entre 0 y 1, lo mismo que ya hicimos con los atributos ordinales. En el caso de los categóricos, sumaremos 1 por cada atributo que tome un valor diferente en los dos inmuebles comparados. Esta forma de medir no toma en cuenta la importancia de los features, y puede representar algún grado de sobre o subdimensionamiento de cada feature. Para intentar reducir el impacto de este problema, usaremos k=10, que es un número pequeño porque representa menos del 1% de las instancias totales. Esperamos que de esa forma todos los 10 vecinos de un inmueble sean muy similares entre sí y eso nos dé un valor representativo. El valor de venta predicho de un inmueble será el promedio de los valores de venta de sus 10 vecinos más cercanos en el conjunto de entrenamiento.

In [54]:
X_train, X_test, target_train, target_test = train_test_split(X_processed, y, test_size=0.1, random_state=42,shuffle=True)

In [55]:
X_train_arr = X_train.to_numpy()
X_test_arr = X_test.to_numpy()
target_train_arr = target_train.to_numpy()
target_test_arr = target_test.to_numpy()
X_train_arr = X_train.astype(float).to_numpy()
X_test_arr = X_test.astype(float).to_numpy()

In [15]:
from sklearn.neighbors import KNeighborsRegressor
from scipy.spatial.distance import pdist, cdist, squareform

In [13]:
kfold = KFold(5)
pesos = 'uniform'

In [16]:
cols_object = X_train.select_dtypes(include='object').columns
print(cols_object)
print(X_train[cols_object].head())

Index([], dtype='object')
Empty DataFrame
Columns: []
Index: [907, 782, 952, 620, 669]


In [17]:
print(X_train.dtypes.value_counts())
print(X_train.dtypes.unique())

bool       127
float64     56
int32       34
Name: count, dtype: int64
[dtype('float64') dtype('bool') dtype('int32')]


In [18]:
for col in X_train.columns:
    try:
        X_train[col].astype(float)
    except Exception as e:
        print(col, "->", X_train[col].dtype, "->", e)

In [24]:
pesos = 'distance'
X_train_arr = X_train.to_numpy()
X_test_arr = X_test.to_numpy()
target_train_arr = target_train.to_numpy()
target_test_arr = target_test.to_numpy()
X_train_arr = X_train.astype(float).to_numpy()
X_test_arr = X_test.astype(float).to_numpy()

# Matriz de distancias L1 entre todos los pares del conjunto de entrenamiento

knn = KNeighborsRegressor(n_neighbors=10, metric='precomputed',weights=pesos)

r2_prom = 0 
mse = 0
for train_idx, val_idx in kfold.split(X_train_arr,target_train_arr):
    X_train_fold, X_val_fold = X_train_arr[train_idx], X_train_arr[val_idx]
    target_train_fold, target_val_fold = target_train_arr[train_idx], target_train_arr[val_idx]
    dist_train = squareform(pdist(X_train_fold, metric='cityblock'))
    knn.fit(dist_train,target_train_fold)
    dist_val_fold = cdist(X_val_fold, X_train_fold, metric='cityblock')
    pred_val_fold = knn.predict(dist_val_fold)
    mse += mean_squared_error(target_val_fold,pred_val_fold)
    r2_prom += r2_score(target_val_fold, pred_val_fold)
r2_prom = r2_prom/5
mse = mse/5
print(r2_prom)
print(mse)

0.7420274356883537
1561719698.7461686


In [23]:
k = 10
rng = np.random.default_rng(42)  # fija la semilla para reproducibilidad; sacala si querés que cambie en cada corrida
idx_random = rng.integers(0, X_train_arr.shape[0])

instancia_vec = X_train_arr[idx_random].reshape(1, -1)
precio_instancia = target_train_arr[idx_random]

# Distancia L1 de la instancia elegida a todo el resto del train
dist_a_train = cdist(instancia_vec, X_train_arr, metric='cityblock')[0]

# La instancia pertenece al propio X_train, así que su "vecino" más cercano
# sería ella misma con distancia 0 -> la excluimos
dist_a_train[idx_random] = np.inf

idx_vecinos = np.argsort(dist_a_train)[:k]
distancias_vecinos = dist_a_train[idx_vecinos]

print(f"Instancia elegida (posición {idx_random} en X_train):\n")
print(X_train.iloc[idx_random])
print(f"\nPrecio de venta real: {precio_instancia}\n")
print("=" * 60)

for rank, (i, dist) in enumerate(zip(idx_vecinos, distancias_vecinos), start=1):
    print(f"\nVecino #{rank} (posición {i}, distancia L1 = {dist:.4f}):\n")
    print(X_train.iloc[i])
    print(f"\nPrecio de venta: {target_train_arr[i]}")
    print("-" * 60)

promedio_vecinos = target_train_arr[idx_vecinos].mean()
print(f"\nPromedio de precios de los {k} vecinos (predicción KNN): {promedio_vecinos:.2f}")

Instancia elegida (posición 117 en X_train):

Street             1.0
Alley              0.0
LotShape           1.0
Utilities          1.0
LandSlope          0.5
                  ... 
BsmtFinType_BLQ      0
BsmtFinType_GLQ      1
BsmtFinType_LwQ      0
BsmtFinType_Rec      0
BsmtFinType_Unf      1
Name: 426, Length: 217, dtype: object

Precio de venta real: 275000


Vecino #1 (posición 92, distancia L1 = 12.7231):

Street                  1.0
Alley                   0.0
LotShape           0.666667
Utilities               1.0
LandSlope               0.5
                     ...   
BsmtFinType_BLQ           0
BsmtFinType_GLQ           1
BsmtFinType_LwQ           0
BsmtFinType_Rec           0
BsmtFinType_Unf           1
Name: 208, Length: 217, dtype: object

Precio de venta: 277000
------------------------------------------------------------

Vecino #2 (posición 440, distancia L1 = 13.1730):

Street             1.0
Alley              0.0
LotShape           1.0
Utilities          1.0
LandS

Como predecir un precio con los k-vecinos más cercanos no funciona, el problema puede ser algo que ya sospechábamos: como los atributos son one hot o normalizados entre 0 y 1, puede estar ponderando de manera uniforme atributos con niveles de importancia muy distintos. Por eso, vamos a probar pesarlos con muchos vectores de pesos aleatorios, esperando que tal vez alguna combinación de pesos represente mejor la realidad del problema.

In [46]:
knn = KNeighborsRegressor(n_neighbors=10, metric='precomputed', weights='uniform')

resultados = []  # (índice del vector de pesos, R2 promedio, MSE promedio)

n_features = X_train_arr.shape[1]
n_vectores = 50

rng = np.random.default_rng(42)  # fija la semilla para reproducibilidad; sacala si querés que cambien en cada corrida
pesos_candidatos = rng.uniform(low=0.0, high=10.0, size=(n_vectores, n_features))

print(pesos_candidatos.shape)  # (100, n_features)

for idx_peso, w in enumerate(pesos_candidatos):
    #X_train_w = X_train_arr * w  # escalado elemento a elemento (implementa la L1 ponderada)
    X_train_w = X_train_arr * np.sqrt(w)
    r2_prom = 0
    mse_prom = 0
    error_relativo_prom = 0

    for train_idx, val_idx in kfold.split(X_train_w, target_train_arr):
        X_train_fold, X_val_fold = X_train_w[train_idx], X_train_w[val_idx]
        target_train_fold, target_val_fold = target_train_arr[train_idx], target_train_arr[val_idx]

        dist_train_fold = squareform(pdist(X_train_fold, metric='euclidean')) #poner cityblock para que sea norma 1.
        knn.fit(dist_train_fold, target_train_fold)

        dist_val_fold = cdist(X_val_fold, X_train_fold, metric='euclidean')
        pred_val_fold = knn.predict(dist_val_fold)

        mse_prom += mean_squared_error(target_val_fold, pred_val_fold)
        r2_prom += r2_score(target_val_fold, pred_val_fold)
        error_relativo_fold = np.mean(
                        np.abs(pred_val_fold - target_val_fold) / target_val_fold
                )
        error_relativo_prom += error_relativo_fold

    r2_prom /= 5
    mse_prom /= 5
    error_relativo_prom /= 5

    resultados.append((idx_peso, r2_prom, mse_prom))
    print(f"Pesos #{idx_peso}: R2 promedio = {r2_prom:.4f} | error relativo promedio = {error_relativo_prom:.2f}")

# Mejor vector de pesos según R2
mejor = max(resultados, key=lambda x: x[1])
print(f"\nMejor vector de pesos: #{mejor[0]} con R2 = {mejor[1]:.4f} y MSE = {mejor[2]:.2f}") #el mejor da R^2 = 0.74 para uniform, 0.75 para distance
#los resultados empeoran con la norma 2 respecto de la norma 1, con un máximo de 0.71.

(50, 217)
Pesos #0: R2 promedio = 0.6650 | error relativo promedio = 0.15
Pesos #1: R2 promedio = 0.6841 | error relativo promedio = 0.16
Pesos #2: R2 promedio = 0.6506 | error relativo promedio = 0.16
Pesos #3: R2 promedio = 0.6795 | error relativo promedio = 0.16
Pesos #4: R2 promedio = 0.6757 | error relativo promedio = 0.16
Pesos #5: R2 promedio = 0.6714 | error relativo promedio = 0.16
Pesos #6: R2 promedio = 0.6712 | error relativo promedio = 0.16
Pesos #7: R2 promedio = 0.6762 | error relativo promedio = 0.16
Pesos #8: R2 promedio = 0.6649 | error relativo promedio = 0.16
Pesos #9: R2 promedio = 0.7028 | error relativo promedio = 0.15
Pesos #10: R2 promedio = 0.6654 | error relativo promedio = 0.16
Pesos #11: R2 promedio = 0.6689 | error relativo promedio = 0.16
Pesos #12: R2 promedio = 0.7050 | error relativo promedio = 0.15
Pesos #13: R2 promedio = 0.6742 | error relativo promedio = 0.16
Pesos #14: R2 promedio = 0.7005 | error relativo promedio = 0.15
Pesos #15: R2 promedio = 

Como el enfoque de k-vecinos más cercanos no funcionó, ni siquiera variando los pesos, probamos ahora con una regresión lineal. Ya que lo vamos a hacer sobre la base de datos preprocesada y muchos atributos son one hot de una misma categoría, las variables que vamos a usar están muy correlacionadas. Además de eso creemos que, de los atributos originales, los realmente importantes no son muchos. Por estos motivos, usaremos regularización Lasso, que tiende a dar soluciones con muchos parámetros muy pequeños o nulos.

In [ ]:
from sklearn.linear_model import Lasso

alphas = [0.1, 1, 10, 25, 50, 75, 100,250,500]
resultados_lasso = []  # (alpha, R2 promedio, MSE promedio)

for alpha in alphas:
    lasso = Lasso(alpha=alpha, random_state=42)

    r2_prom = 0
    mse_prom = 0

    for train_idx, val_idx in kfold.split(X_train_arr, target_train_arr):
        X_train_fold, X_val_fold = X_train_arr[train_idx], X_train_arr[val_idx]
        target_train_fold, target_val_fold = target_train_arr[train_idx], target_train_arr[val_idx]

        lasso.fit(X_train_fold, target_train_fold)
        pred_val_fold = lasso.predict(X_val_fold)

        mse_prom += mean_squared_error(target_val_fold, pred_val_fold)
        r2_prom += r2_score(target_val_fold, pred_val_fold)

    r2_prom /= 5
    mse_prom /= 5

    resultados_lasso.append((alpha, r2_prom, mse_prom))
    print(f"alpha = {alpha}: R2 promedio = {r2_prom:.4f} | MSE promedio = {mse_prom:.2f}")

mejor_alpha = max(resultados_lasso, key=lambda x: x[1])
print(f"\nMejor alpha: {mejor_alpha[0]} con R2 = {mejor_alpha[1]:.4f} y MSE = {mejor_alpha[2]:.2f}")

c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.019e+11, tolerance: 6.409e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.928e+10, tolerance: 6.012e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.007e+10, tolerance: 

alpha = 0.1: R2 promedio = 0.8082 | MSE promedio = 1195055226.66


c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.696e+10, tolerance: 6.409e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.838e+10, tolerance: 6.012e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.077e+10, tolerance: 

alpha = 1: R2 promedio = 0.8082 | MSE promedio = 1196680001.11
alpha = 10: R2 promedio = 0.8206 | MSE promedio = 1121453143.39
alpha = 25: R2 promedio = 0.8309 | MSE promedio = 1060347608.65
alpha = 50: R2 promedio = 0.8368 | MSE promedio = 1023254623.93
alpha = 75: R2 promedio = 0.8397 | MSE promedio = 1005268926.89
alpha = 100: R2 promedio = 0.8410 | MSE promedio = 996541637.80
alpha = 250: R2 promedio = 0.8383 | MSE promedio = 1004075333.11
alpha = 500: R2 promedio = 0.8240 | MSE promedio = 1083334832.70

Mejor alpha: 100 con R2 = 0.8410 y MSE = 996541637.80


Aunque los resultados superan visiblemente a los k-vecinos más cercanos, lo que puede ser señal de que estamos en el camino correcto, vemos que en general no convergió, lo que se demuestra con valores de tolerance altos y duality gaps mucho mayores. Esto posiblemente se deba a la enorme diferencia de escala entre las variables normalizadas y el target (precios de venta de inmuebles en dólares contra valores entre 0 y 1). Por eso, vamos a intentar de nuevo escalando también el target.

In [43]:
from sklearn.preprocessing import MinMaxScaler

target_scaler = MinMaxScaler()
target_train_scaled = target_scaler.fit_transform(target_train_arr.reshape(-1, 1)).flatten()

alphas = [0.0001, 0.001, 0.01, 0.1]
resultados_lasso = []  # (alpha, R2 promedio, MSE promedio en dólares²)

for alpha in alphas:
    lasso = Lasso(alpha=alpha, random_state=42, max_iter=100000)

    r2_prom = 0
    mse_prom_dolares = 0
    error_relativo_prom = 0

    for train_idx, val_idx in kfold.split(X_train_arr, target_train_scaled):
        X_train_fold, X_val_fold = X_train_arr[train_idx], X_train_arr[val_idx]
        target_train_fold_scaled = target_train_scaled[train_idx]
        target_val_fold_scaled = target_train_scaled[val_idx]

        lasso.fit(X_train_fold, target_train_fold_scaled)
        pred_val_fold_scaled = lasso.predict(X_val_fold)

        # Des-escalar predicción y target real para reportar en dólares
        pred_val_fold_dolares = target_scaler.inverse_transform(
            pred_val_fold_scaled.reshape(-1, 1)
        ).flatten()
        target_val_fold_dolares = target_scaler.inverse_transform(
            target_val_fold_scaled.reshape(-1, 1)
        ).flatten()
        error_relativo_fold = np.mean(
                np.abs(pred_val_fold_dolares - target_val_fold_dolares) / target_val_fold_dolares
        )

        mse_prom_dolares += mean_squared_error(target_val_fold_dolares, pred_val_fold_dolares)
        r2_prom += r2_score(target_val_fold_dolares, pred_val_fold_dolares)
        error_relativo_prom += error_relativo_fold

    r2_prom /= 5
    mse_prom_dolares /= 5
    error_relativo_prom /= 5
    resultados_lasso.append((alpha, r2_prom, mse_prom_dolares, error_relativo_prom))
    print(f"alpha = {alpha}: R2 promedio = {r2_prom:.4f} | Error relativo promedio = {error_relativo_prom:.2f}")

mejor_alpha = max(resultados_lasso, key=lambda x: x[1])
print(f"\nMejor alpha: {mejor_alpha[0]} con R2 = {mejor_alpha[1]:.4f} y MSE = {mejor_alpha[2]:.2f}")

alpha = 0.0001: R2 promedio = 0.8393 | Error relativo promedio = 0.11
alpha = 0.001: R2 promedio = 0.8091 | Error relativo promedio = 0.12
alpha = 0.01: R2 promedio = 0.4030 | Error relativo promedio = 0.25
alpha = 0.1: R2 promedio = -0.0054 | Error relativo promedio = 0.36

Mejor alpha: 0.0001 con R2 = 0.8393 y MSE = 1007722655.80


Para los primeros coeficientes de alpha probados: [0.1, 1, 10, 100], esto da un R^2 de -0.0054, muy cercano a 0. Esto es el resultado cuando se predice siempre el promedio de todas las instancias. El motivo, que se confirma con el siguiente bloque de código, es que todos los coeficientes fueron llevados a 0 porque los alpha utilizados (iguales a los de antes) conllevaron una regularización excesiva (valores alpha entre 0.1 y 100). Buscaremos arreglarlo reduciendo la escala.

Con valores alpha = 0.0001, 0.001, 0.01, 0.1, se tiene valores R^2 promedio de 0.84, 0.81, 0.4030, -0.0054 respectivamente. Se ve que el menor alpha de todos no funciona mal, pero ver el deterioro de los resultados con el crecimiento de este parámetro nos invita a buscar el mejor en un rango aún más chico: 0.0000001, 0.000001, 0.00001, 0.0001. Todos en este rango superan el R^2 de 0.80, pero el mejor sigue siendo 0.0001, por lo que buscaremos más profundamente en este orden de magnitud. 

Como resultado, el mejor R^2 (0.84) se obtiene con alpha = 0.0002, pero no aporta una diferencia significativa con respecto a 0.0001 o 0.0003. En todos los k*10^{-4}, con k entre 0 y 9, el error relativo promedio da 0.11 o 0.12. Para comparar esta nueva métrica, corremos de nuevo los alpha anteriores que nos daban peor. Para los originales, entre 0.1 y 100, los errores relativos promedio dan 0.36.

In [36]:
print((lasso.coef_ != 0).sum(), "de", len(lasso.coef_), "coeficientes distintos de cero")
print(lasso.intercept_)

36 de 217 coeficientes distintos de cero
-0.09057142779948252


Vemos que el mejor método de regresión utilizado en este trabajo es el HistBoostingRegressor, seguido muy de cerca por la regresión lineal con regularización Lasso con alpha del orden de magnitud adecuado. Finalmente evaluamos ambos con todo el conjunto de entrenamiento para ver los resultados en el conjunto de test.

In [53]:
pred_val_fold = []
target_val_fold = []
kfold = KFold(n_splits=5)
for prof_max in lista_max_depth:
    for min_leaf in lista_min_samples_leaf:
        for max_feat in lista_max_features:
            model = HistGradientBoostingRegressor(loss='squared_error',max_depth=prof_max,min_samples_leaf=min_leaf,
                                              max_features=max_feat,random_state=42, categorical_features=categoricas)
            model.fit(X_train,target_train)
            pred_val = model.predict(X_test)
            error_relativo = np.mean(np.abs(pred_val - target_test) / target_test)
            r2= r2_score(target_test, pred_val)
            print('etapa')
            print(prof_max,min_leaf,max_feat,'r2 promedio = ',r2_prom,error_relativo_prom)

etapa
3 1 0.25 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 1 0.5 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 1 0.75 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 1 1.0 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 5 0.25 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 5 0.5 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 5 0.75 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 5 1.0 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 25 0.25 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 25 0.5 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 25 0.75 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 25 1.0 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 50 0.25 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 50 0.5 r2 promedio =  0.6853008030134451 0.15366540833097503
etapa
3 50 0.75 r2 promedio =  0.68

Notamos que el HistBoostingRegressor sobreajustó, ya que se observa de forma consistente un mayor error relativo que durante la etapa de desarrollo. Creemos que no va a ocurrir con la regresión Lasso, ya que convierte muchos parámetros en 0 para prevenir sobreajustes.

In [56]:
alpha = 0.0002

lasso_final = Lasso(alpha=alpha, random_state=42, max_iter=100000)
lasso_final.fit(X_train_arr, target_train_scaled)

# Predicción sobre test, en la escala [0,1] del target
pred_test_scaled = lasso_final.predict(X_test_arr)

# Des-escalar con el mismo target_scaler ajustado sobre target_train_arr
# (NO se vuelve a fitear con datos de test, para no filtrar información)
pred_test_dolares = target_scaler.inverse_transform(
    pred_test_scaled.reshape(-1, 1)
).flatten()
target_test_dolares = target_test_arr  # ya está en dólares, sin escalar

mse_test = mean_squared_error(target_test_dolares, pred_test_dolares)
r2_test = r2_score(target_test_dolares, pred_test_dolares)
error_relativo_test = np.mean(
    np.abs(pred_test_dolares - target_test_dolares) / target_test_dolares
)

print(f"R2 en test = {r2_test:.4f}")
print(f"MSE en test (dólares^2) = {mse_test:.2f}")
print(f"Error relativo promedio en test = {error_relativo_test:.4f} ({error_relativo_test*100:.2f}%)")

R2 en test = 0.8535
MSE en test (dólares^2) = 1338486787.17
Error relativo promedio en test = 0.1083 (10.83%)


El error relativo promedio se mantiene en 0.11 aproximadamente, es decir que mantiene el buen resultado de la etapa de desarrollo. Por lo tanto, elegimos la regularización Lasso con parámetro alpha = 0.0002 como modelo final.